## Sending a test event to the Event Hub

In [0]:

from azure.eventhub import EventHubProducerClient, EventData

import json

# Get secret value from Key Vault
eventhub_connection_string = dbutils.secrets.get(scope = "key-vault-scope", key = "eventhub-connection-string")

EVENT_HUB_NAME = "weatherstreamingeventhub"

# Initialize the eventhub producer
producer = EventHubProducerClient.from_connection_string(eventhub_connection_string, eventhub_name=EVENT_HUB_NAME)

# Function to send events to Event Hub
def send_event(event):
    event_data_batch = producer.create_batch()
    event_data_batch.add(EventData(json.dumps(event)))
    producer.send_batch(event_data_batch)

# Sample JSON envent
event = {
    "event_id": 2222,
    "event_name": "Key Vault Test"
}

# Send the event
send_event(event)

# Close the producer
producer.close()

### API Testing



In [0]:
import requests
import json

# Get secret value from Key Vault
weatherapikey = dbutils.secrets.get(scope="key-vault-scope", key="WeatherAPIKey")
location = "Noida"

base_url = "https://api.weatherapi.com/v1/"

current_weather_url = f"{base_url}/current.json"

params = {
    'key': weatherapikey,
    'q': location,
}

response = requests.get(current_weather_url, params=params)
if response.status_code == 200:
    current_weather = response.json()
    print("Current weather:")
    print(json.dumps(current_weather, indent=3))
else:
    print(f"Error: {response.status_code}, {response.text}")



Current weather:
{
   "location": {
      "name": "Noida",
      "region": "Uttar Pradesh",
      "country": "India",
      "lat": 28.57,
      "lon": 77.32,
      "tz_id": "Asia/Kolkata",
      "localtime_epoch": 1776766725,
      "localtime": "2026-04-21 15:48"
   },
   "current": {
      "last_updated_epoch": 1776766500,
      "last_updated": "2026-04-21 15:45",
      "temp_c": 39.1,
      "temp_f": 102.4,
      "is_day": 1,
      "condition": {
         "text": "Sunny",
         "icon": "//cdn.weatherapi.com/weather/64x64/day/113.png",
         "code": 1000
      },
      "wind_mph": 14.1,
      "wind_kph": 22.7,
      "wind_degree": 294,
      "wind_dir": "WNW",
      "pressure_mb": 1004.0,
      "pressure_in": 29.65,
      "precip_mm": 0.0,
      "precip_in": 0.0,
      "humidity": 14,
      "cloud": 0,
      "feelslike_c": 38.6,
      "feelslike_f": 101.5,
      "windchill_c": 41.0,
      "windchill_f": 105.9,
      "heatindex_c": 41.5,
      "heatindex_f": 106.7,
      "dewpoin

### Complete code for getting weather data

In [0]:
import requests
import json

# Function to handle the API response
def handle_response(response):
  if response.status_code == 200:
    return response.json()
  else:
    return f"Error: {response.status_code}, {response.text}"

# Function to get current weather and air quality data
def get_current_weather(base_url, api_key, location):
  current_weather_url = f"{base_url}/current.json"
  params = {
    'key': api_key,
    'q': location,
    'aqi': 'yes'
  }
  response = requests.get(current_weather_url, params=params)
  return handle_response(response)

# Function to get forecast data
def get_forecast_weather(base_url, api_key, location, days):
  forecast_url = f"{base_url}/forecast.json"
  params = {
    'key': api_key,
    'q': location,
    'days': days,
  }
  response = requests.get(forecast_url, params=params)
  return handle_response(response)


# Function to get alerts
def get_alerts(base_url, api_key, location):
  alerts_url = f"{base_url}/alerts.json"
  params = {
    'key': api_key,
    'q': location,
    'alerts': 'yes'
  }
  response = requests.get(alerts_url, params=params)
  return handle_response(response)


# Flatten and merge the data
def flatten_data(current_weather, forecast_weather, alerts):
  location_data = current_weather.get("location", {})
  current = current_weather.get("current", {})
  condition = current.get("condition", {})
  air_quality = current.get("air_quality", {})
  forecast = forecast_weather.get("forecast", {}).get("forecastday", [])
  alert_list = alerts.get("alerts", {}).get("alert", [])

  flattened_data = {
    'name': location_data.get("name"),
    'region': location_data.get("region"),
    'country': location_data.get("country"),
    'lat': location_data.get("lat"),
    'lon': location_data.get("lon"),
    'localtime': location_data.get("localtime"),
    'temp_c': current.get('temp_c'), 
    'is_day': current.get('is_day'),
    'condition_text': condition.get('text'),
    'condition_icon': condition.get('icon'),
    'wind_kph': current.get('wind_kph'),
    'wind_degree': current.get('wind_degree'),
    'wind_dir': current.get('wind_dir'),
    'pressure_mb': current.get('pressure_mb'),
    'precip_mm': current.get('precip_mm'),
    'humidity': current.get('humidity'),
    'cloud': current.get('cloud'),
    'feelslike_c': current.get('feelslike_c'),
    'uv': current.get('uv'),

    'air_quality':{
      'co': air_quality.get('co'),
      'no2': air_quality.get('no2'),
      'o3': air_quality.get('o3'),
      'so2': air_quality.get('so2'),
      'pm2_5': air_quality.get('pm2_5'),
      'pm10': air_quality.get('pm10'),
      'us-epa-index': air_quality.get('us-epa-index'), 
      'gb-defra-index': air_quality.get('gb-defra-index'),
    }, 

    'alerts': [
      {
        'headline': alert.get('headline'),
        'severity': alert.get('severity'),
        'description': alert.get('description'),
        'instruction': alert.get('instruction')
      }
      for alert in alert_list
    ],

    'forecast': [
      {
        'date': day.get('date'),
        'maxtemp_c': day.get('day', {}).get('maxtemp_c'),
        'mintemp_c': day.get('day', {}).get('mintemp_c'),
        'condition': day.get('day', {}).get('condition', {}).get('text')
      }
      for day in forecast
    ]
  }

  return flattened_data


# Main function
def fetch_weather_data():
  base_url = "https://api.weatherapi.com/v1/"
  location = "Noida"
  weatherapikey = dbutils.secrets.get(scope="key-vault-scope", key="weatherapikey")

  # Get data from API
  current_weather = get_current_weather(base_url, weatherapikey, location)
  forecast_weather = get_forecast_weather(base_url, weatherapikey, location, 3)
  alerts = get_alerts(base_url, weatherapikey, location)

  # Flatten and merge data
  merged_data = flatten_data(current_weather, forecast_weather, alerts)
  print("Weather data:", json.dumps(merged_data, indent=3))

# Call the main function
fetch_weather_data()
   

Weather data: {
   "name": "Noida",
   "region": "Uttar Pradesh",
   "country": "India",
   "lat": 28.57,
   "lon": 77.32,
   "localtime": "2026-04-21 16:30",
   "temp_c": 38.3,
   "is_day": 1,
   "condition_text": "Sunny",
   "condition_icon": "//cdn.weatherapi.com/weather/64x64/day/113.png",
   "wind_kph": 23.4,
   "wind_degree": 293,
   "wind_dir": "WNW",
   "pressure_mb": 1004.0,
   "precip_mm": 0.0,
   "humidity": 15,
   "cloud": 0,
   "feelslike_c": 37.5,
   "uv": 1.7,
   "air_quality": {
      "co": 328.85,
      "no2": 5.35,
      "o3": 170.0,
      "so2": 14.85,
      "pm2_5": 97.25,
      "pm10": 937.75,
      "us-epa-index": 4,
      "gb-defra-index": 10
   },
   "alerts": [],
   "forecast": [
      {
         "date": "2026-04-21",
         "maxtemp_c": 41.0,
         "mintemp_c": 30.2,
         "condition": "Sunny"
      },
      {
         "date": "2026-04-22",
         "maxtemp_c": 41.4,
         "mintemp_c": 31.1,
         "condition": "Sunny"
      },
      {
         "

## Sending the complete weather data to event hub

In [0]:
import requests
import json
from azure.eventhub import EventHubProducerClient, EventData

# Get secret value from Key Vault
eventhub_connection_string = dbutils.secrets.get(scope="key-vault-scope", key="eventhub-connection-string")

EVENT_HUB_NAME = "weatherstreamingeventhub"

# Initialize the eventhub producer
producer = EventHubProducerClient.from_connection_string(eventhub_connection_string, eventhub_name=EVENT_HUB_NAME)

# Function to send events to Event Hub
def send_event(event):
    event_data_batch = producer.create_batch()
    event_data_batch.add(EventData(json.dumps(event)))
    producer.send_batch(event_data_batch)

# Function to handle the API response
def handle_response(response):
  if response.status_code == 200:
    return response.json()
  else:
    return f"Error: {response.status_code}, {response.text}"

# Function to get current weather and air quality data
def get_current_weather(base_url, api_key, location):
  current_weather_url = f"{base_url}/current.json"
  params = {
    'key': api_key,
    'q': location,
    'aqi': 'yes'
  }
  response = requests.get(current_weather_url, params=params)
  return handle_response(response)

# Function to get forecast data
def get_forecast_weather(base_url, api_key, location, days):
  forecast_url = f"{base_url}/forecast.json"
  params = {
    'key': api_key,
    'q': location,
    'days': days,
  }
  response = requests.get(forecast_url, params=params)
  return handle_response(response)

# Function to get alerts
def get_alerts(base_url, api_key, location):
  alerts_url = f"{base_url}/alerts.json"
  params = {
    'key': api_key,
    'q': location,
    'alerts': 'yes'
  }
  response = requests.get(alerts_url, params=params)
  return handle_response(response)


# Flatten and merge the data
def flatten_data(current_weather, forecast_weather, alerts):
  location_data = current_weather.get("location", {})
  current = current_weather.get("current", {})
  condition = current.get("condition", {})
  air_quality = current.get("air_quality", {})
  forecast = forecast_weather.get("forecast", {}).get("forecastday", [])
  alert_list = alerts.get("alerts", {}).get("alert", [])

  flattened_data = {
    'name': location_data.get("name"),
    'region': location_data.get("region"),
    'country': location_data.get("country"),
    'lat': location_data.get("lat"),
    'lon': location_data.get("lon"),
    'localtime': location_data.get("localtime"),
    'temp_c': current.get('temp_c'), 
    'is_day': current.get('is_day'),
    'condition_text': condition.get('text'),
    'condition_icon': condition.get('icon'),
    'wind_kph': current.get('wind_kph'),
    'wind_degree': current.get('wind_degree'),
    'wind_dir': current.get('wind_dir'),
    'pressure_mb': current.get('pressure_mb'),
    'precip_mm': current.get('precip_mm'),
    'humidity': current.get('humidity'),
    'cloud': current.get('cloud'),
    'feelslike_c': current.get('feelslike_c'),
    'uv': current.get('uv'),

    'air_quality':{
      'co': air_quality.get('co'),
      'no2': air_quality.get('no2'),
      'o3': air_quality.get('o3'),
      'so2': air_quality.get('so2'),
      'pm2_5': air_quality.get('pm2_5'),
      'pm10': air_quality.get('pm10'),
      'us-epa-index': air_quality.get('us-epa-index'), 
      'gb-defra-index': air_quality.get('gb-defra-index'),
    }, 

    'alerts': [
      {
        'headline': alert.get('headline'),
        'severity': alert.get('severity'),
        'description': alert.get('description'),
        'instruction': alert.get('instruction')
      }
      for alert in alert_list
    ],

    'forecast': [
      {
        'date': day.get('date'),
        'maxtemp_c': day.get('day', {}).get('maxtemp_c'),
        'mintemp_c': day.get('day', {}).get('mintemp_c'),
        'condition': day.get('day', {}).get('condition', {}).get('text')
      }
      for day in forecast
    ]
  }

  return flattened_data


# Main function
def fetch_weather_data():
  base_url = "https://api.weatherapi.com/v1/"
  location = "Noida"
  weatherapikey = dbutils.secrets.get(scope="key-vault-scope", key="weatherapikey")

  # Get data from API
  current_weather = get_current_weather(base_url, weatherapikey, location)
  forecast_weather = get_forecast_weather(base_url, weatherapikey, location, 3)
  alerts = get_alerts(base_url, weatherapikey, location)

  # Flatten and merge data
  merged_data = flatten_data(current_weather, forecast_weather, alerts)
  # Send the weather data to the Event Hub
  send_event(merged_data)

# Call the main function,
fetch_weather_data()


## Sending the weather data as a continuous stream in 30s

In [0]:
from datetime import datetime 
import requests
import json
import time
from azure.eventhub import EventHubProducerClient, EventData

# Get secret value from Key Vault
eventhub_connection_string = dbutils.secrets.get(scope="key-vault-scope", key="eventhub-connection-string")
weatherapikey = dbutils.secrets.get(scope="key-vault-scope", key="weatherapikey")

EVENT_HUB_NAME = "weatherstreamingeventhub"

def send_event(event):
    producer = EventHubProducerClient.from_connection_string(eventhub_connection_string, eventhub_name=EVENT_HUB_NAME)
    try:
        event_data_batch = producer.create_batch()
        event_data_batch.add(EventData(json.dumps(event)))
        producer.send_batch(event_data_batch)
    finally:
        producer.close()

def handle_response(response):
    if response.status_code == 200:
        return response.json()
    else:
        return f"Error: {response.status_code}, {response.text}"

def get_current_weather(base_url, api_key, location):
    response = requests.get(f"{base_url}/current.json", params={'key': api_key, 'q': location, 'aqi': 'yes'})
    return handle_response(response)

def get_forecast_weather(base_url, api_key, location, days):
    response = requests.get(f"{base_url}/forecast.json", params={'key': api_key, 'q': location, 'days': days})
    return handle_response(response)

def get_alerts(base_url, api_key, location):
    response = requests.get(f"{base_url}/alerts.json", params={'key': api_key, 'q': location, 'alerts': 'yes'})
    return handle_response(response)

def flatten_data(current_weather, forecast_weather, alerts):
    location_data = current_weather.get("location", {})
    current = current_weather.get("current", {})
    condition = current.get("condition", {})
    air_quality = current.get("air_quality", {})
    forecast = forecast_weather.get("forecast", {}).get("forecastday", [])
    alert_list = alerts.get("alerts", {}).get("alert", [])

    return {
        'name': location_data.get("name"),
        'region': location_data.get("region"),
        'country': location_data.get("country"),
        'lat': location_data.get("lat"),
        'lon': location_data.get("lon"),
        'localtime': location_data.get("localtime"),
        'temp_c': current.get('temp_c'),
        'is_day': current.get('is_day'),
        'condition_text': condition.get('text'),
        'condition_icon': condition.get('icon'),
        'wind_kph': current.get('wind_kph'),
        'wind_degree': current.get('wind_degree'),
        'wind_dir': current.get('wind_dir'),
        'pressure_mb': current.get('pressure_mb'),
        'precip_mm': current.get('precip_mm'),
        'humidity': current.get('humidity'),
        'cloud': current.get('cloud'),
        'feelslike_c': current.get('feelslike_c'),
        'uv': current.get('uv'),
        'air_quality': {
            'co': air_quality.get('co'),
            'no2': air_quality.get('no2'),
            'o3': air_quality.get('o3'),
            'so2': air_quality.get('so2'),
            'pm2_5': air_quality.get('pm2_5'),
            'pm10': air_quality.get('pm10'),
            'us-epa-index': air_quality.get('us-epa-index'),
            'gb-defra-index': air_quality.get('gb-defra-index'),
        },
        'alerts': [
            {
                'headline': alert.get('headline'),
                'severity': alert.get('severity'),
                'description': alert.get('description'),
                'instruction': alert.get('instruction')
            }
            for alert in alert_list
        ],
        'forecast': [
            {
                'date': day.get('date'),
                'maxtemp_c': day.get('day', {}).get('maxtemp_c'),
                'mintemp_c': day.get('day', {}).get('mintemp_c'),
                'condition': day.get('day', {}).get('condition', {}).get('text')
            }
            for day in forecast
        ]
    }

def fetch_and_send():
    base_url = "https://api.weatherapi.com/v1"
    location = "Noida"

    current_weather = get_current_weather(base_url, weatherapikey, location)
    forecast_weather = get_forecast_weather(base_url, weatherapikey, location, 3)
    alerts = get_alerts(base_url, weatherapikey, location)

    merged_data = flatten_data(current_weather, forecast_weather, alerts)
    send_event(merged_data)
    print(f"Event sent at {datetime.now()}")

# ✅ Run every 30 seconds indefinitely
while True:
    try:
        fetch_and_send()
    except Exception as e:
        print(f"Error: {e}")
    time.sleep(30)

Event sent at 2026-04-23 19:19:49.248339
Event sent at 2026-04-23 19:20:20.242128
Event sent at 2026-04-23 19:20:51.291887
Event sent at 2026-04-23 19:21:22.297608


com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:139)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:139)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can